# Session 2 — Python for AI engineers

**Goal:** build a typed `Document` loader with real error handling, then compare it with the package's version.

In [ ]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
    print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

In [ ]:
from bootcamp_agent.checks import check, review

## 1. The data shape first

A `dataclass` is a contract: named, typed fields that cross module boundaries. No bare dicts.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class MiniDocument:
    doc_id: str
    title: str
    text: str


sample = MiniDocument(doc_id="demo", title="Demo", text="Hello corpus.")
print(sample)

## 2. Exercise: a loader with failure modes

**Context.** A loader that returns half a document on bad input is worse than one that refuses. **Errors are part of the contract.**

**Instructions.**

1. Failure mode 1 is done: a missing file raises `ValueError` naming the path.
2. Add failure mode 2: the first line must be `# <title>`; otherwise raise `ValueError` naming the file.
3. Build the `MiniDocument`: `doc_id` is the file stem, `title` is the first line without `# `, `text` is the rest.
4. Run the check. It writes a good file, a bad file, and asks for a missing one.

In [ ]:
from pathlib import Path


def load_mini(path: Path) -> MiniDocument:
    if not path.is_file():
        raise ValueError(f"no such file: {path}")  # failure mode 1, done
    lines = path.read_text(encoding="utf-8").splitlines()
    # TODO(you): failure mode 2, the first line must be "# <title>"
    # TODO(you): return MiniDocument(doc_id=..., title=..., text=...)
    raise NotImplementedError


try:
    load_mini(Path("does-not-exist.md"))  # failure mode 1 runs today
except ValueError as error:
    print(f"refused: {error}")

**Expected output** (yours may differ in wording, not in shape):

```
refused: no such file: does-not-exist.md
Class-by-Class Curriculum
✅ ch02-e1 passed
```

In [ ]:
check("ch02-e1", load_mini)

## 3. Compare with the real loader

The package's loader does the same job with an HTML-comment header and typed `CorpusError`s. Read `src/bootcamp_agent/documents.py`, then load the teaching corpus.

In [ ]:
from bootcamp_agent.documents import load_corpus

documents = load_corpus(CORPUS_DIR)
for doc in documents:
    print(f"{doc.doc_id:22} {doc.title:22} tags={list(doc.tags)}")

## 4. Exercise: an index over the corpus

**Context.** Retrieval (week 2) starts with indexes. The simplest one maps each tag to the documents that carry it.

**Instructions.**

1. The loop skeleton is done. Fill the body: append `doc.doc_id` under each of its tags.
2. Sort every list so the output is deterministic.
3. Print the index, then run the check. It compares with the real corpus.

In [ ]:
tag_index: dict[str, list[str]] = {}
for doc in documents:
    for tag in doc.tags:
        pass  # TODO(you): tag_index.setdefault(tag, []).append(...)
# TODO(you): sort each list

for tag, ids in sorted(tag_index.items()):
    print(f"{tag:16} {ids}")

**Expected output** (yours may differ in wording, not in shape):

```
agent            ['agent-loops']
agents           ['mcp-overview']
...
tools            ['agent-loops', 'mcp-overview']
...
✅ ch02-e2 passed
```

In [ ]:
check("ch02-e2", tag_index)

## Exit ticket

Homework: add three more tests to your loader (empty file? title-only file? wrong extension?) and write a short CONTRIBUTING note on how to run this project locally with uv.

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [ ]:
review("ch02")